In [14]:
from CSVtoSQL import CSVFolderToSQL
CSVFolderToSQL("../synthea/output/csv", "medical_data.db").csv_to_sql()

In [ ]:
import pandas as pd
import numpy as np
from SQL_query import SQLQuery
SQL = SQLQuery("medical_data.db")
query = """
SELECT 
    t1.Id,
    DATE(t1.BIRTHDATE) AS BIRTHDATE,
    DATE(t1.DEATHDATE) AS DEATHDATE,
    t1.RACE,
    t1.ETHNICITY,
    t1.GENDER,
    t1.COUNTY,
    t1.HEALTHCARE_EXPENSES,
    t1.HEALTHCARE_COVERAGE,
    t1.INCOME,
    DATE(t2.PREGNANCY_START) AS PREGNANCY_START
FROM patients as t1
INNER JOIN (
    SELECT 
        PATIENT,
        MAX(START) AS PREGNANCY_START
    FROM encounters
    WHERE DESCRIPTION = 'Prenatal initial visit (regime/therapy)'
    GROUP BY PATIENT
) as t2 ON t1.Id = t2.PATIENT
"""
LAfemMed = SQL.execute_query(query)
LAfemMed['BIRTHDATE'] = pd.to_datetime(LAfemMed['BIRTHDATE'])
LAfemMed['DEATHDATE'] = pd.to_datetime(LAfemMed['DEATHDATE'])
LAfemMed['PREGNANCY_START'] = pd.to_datetime(LAfemMed['PREGNANCY_START'])
LAfemMed['Days_Pregancy_to_Death'] = (LAfemMed['DEATHDATE'] - LAfemMed['PREGNANCY_START']).dt.days
LAfemMed['Maternal_Mortality'] = np.where(LAfemMed['Days_Pregancy_to_Death'] <= 292, 1, 0)